# Anomaly-Based IDS Poisoning Robustness — SWaT Dataset
## PAPER RUN v2 — Narval cell-by-cell edition

**Authors:** Mustafa Umut Özbek et al.

This is the Narval-adapted version of the paper-run notebook. Scientific
cells are **identical** to the original. Only the Colab/Drive cells (1.1,
1.4, 1.5) have been rewritten to use Narval env vars and the venv-managed
Python environment.

### How to run this notebook on Narval

1. Allocate a GPU interactively (from a login node):
   ```
   salloc --account=def-liyang --gres=gpu:a100:1 \
          --cpus-per-task=6 --mem=48G --time=6:00:00
   ```
2. Inside the allocation, activate the project venv:
   ```
   cd ~/projects/def-liyang/$USER/narval_swat_run
   module load StdEnv/2023 python/3.11 scipy-stack cuda/12.2
   source venv/bin/activate
   ```
3. Open this notebook via VS Code Remote-SSH to Narval, select the venv
   Python interpreter, and step through the cells.

### What's different vs the original notebook

- **No `!pip install`** (cell 1.1 is a no-op — the venv already has everything)
- **No Drive mount / path autodiscovery** (cell 1.4 reads `$SWAT_DATA_DIR` and
  `$SWAT_OUTPUT_DIR` directly)
- **No checkpoint migration** from v16_* or final_paper_run folders — this
  run is a clean slate
- Everything else: preprocess, splits, models, attacks, grid, tables,
  figures, diagnostics — unchanged

### Paper-run grid

| Stage | Runs |
|---|---|
| Clean baselines (§5.1) | 12 models × 3 seeds = 36 |
| LSTM-AE diagnostic (§5.1b) | 1 |
| Poisoning grid (§6.2) | 12 × 3 × 4 × 3 = 432 |
| **Total** | **468** |

A full cell-by-cell run takes roughly **3–6 hours on an A100 40GB**.
If your `salloc` walltime is shorter, the checkpoints let you resume
— restart VS Code and rerun cell 6.2; it will skip finished combos.

---


---
## 1. Setup


In [ ]:
# 1.1 Dependency check (Narval edition)
# The venv is created once via env/setup_venv.sh; we just verify that every
# package we need is importable. No `pip install` at notebook time.
import importlib, sys
for mod in ["numpy", "pandas", "scipy", "sklearn", "matplotlib", "seaborn",
            "torch", "pyod"]:
    importlib.import_module(mod)
print("Python :", sys.version.split()[0])
print("All dependencies available.")


In [ ]:
# 1.2 Imports

import os, sys, time, warnings, json, gc, random, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product as iterproduct
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    confusion_matrix, roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve
)
from sklearn.decomposition import PCA as SkPCA

# PyOD models
from pyod.models.iforest import IForest
from pyod.models.ocsvm import OCSVM
from pyod.models.lof import LOF
from pyod.models.cblof import CBLOF
from pyod.models.knn import KNN
from pyod.models.hbos import HBOS
from pyod.models.pca import PCA as PCA_AD
from pyod.models.mcd import MCD
from pyod.models.abod import ABOD
from pyod.models.sod import SOD

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print('All imports loaded.')

# ─── pyod 2.0.3 + sklearn 1.6+ sklearn tags shim (bulletproof, runs at import time) ───
def _pyod_sklearn_tags(self):
    class _Tags:
        estimator_type = "outlier_detector"
        requires_fit = True
        non_deterministic = False
        array_api_support = False
        no_validation = False
        _skip_test = False
        transformer_tags = None
        classifier_tags = None
        regressor_tags = None
        class input_tags:
            sparse = False; allow_nan = True; pairwise = False
            string = False; categorical = False; dict = False
            one_d_array = False; three_d_array = False
        class target_tags:
            required = False; one_d_labels = False; two_d_labels = False
            positive_only = False; multi_output = False; single_output = True
    return _Tags()
for _cls in (IForest, OCSVM, LOF, CBLOF, KNN, HBOS, PCA_AD, MCD, ABOD, SOD):
    _cls.__sklearn_tags__ = _pyod_sklearn_tags


In [ ]:
# 1.3 Configuration — PAPER RUN (tuned params baked in)
CONFIG = {
    'POISON_RATES':  [0.01, 0.03, 0.05, 0.10],
    'SEEDS':         [42, 123, 456],
    'ATTACKS':       ['random_flip', 'targeted_flip', 'feature_noise'],
    'MODELS':        ['iforest', 'svm', 'lof', 'cluster', 'knn', 'histogram',
                      'pca', 'mcd', 'abod', 'sod', 'autoencoder', 'lstm_ae'],
    'TEST_SIZE':     0.15,
    'VAL_SIZE':      0.15,
    'CONTAMINATION': 0.05,

    # ── Scalability ──
    'HEAVY_MODELS':      ['svm', 'lof', 'knn', 'abod', 'sod', 'mcd'],
    'MAX_TRAIN_SAMPLES': 50000,   # subsample cap for heavy models

    # ── Autoencoder (V3) ──
    'AE_HIDDEN_DIMS': [256, 128, 64],
    'AE_EPOCHS':      100,
    'AE_LR':          0.0005,
    'AE_DROPOUT':     0.1,
    'AE_PATIENCE':    15,

    # ── LSTM-Autoencoder ──
    'LSTM_AE_WINDOW':    20,
    'LSTM_AE_HIDDEN':    128,
    'LSTM_AE_EPOCHS':    50,
    'LSTM_AE_LR':        0.0005,
    'LSTM_AE_DROPOUT':   0.2,
    'LSTM_AE_PATIENCE':  15,

    # ── Feature-noise injection ──
    'NOISE_SIGMA': 0.15,
}

# ─────────────────────────────────────────────────────────────────────────
# TUNED PARAMETERS — from Phase 1 sensitivity study, Phase 2 confirmed
# ─────────────────────────────────────────────────────────────────────────
# These are the ONLY non-default hyperparameters in this paper run.
# The other 8 PyOD models (iforest, lof, cluster, knn, histogram, mcd, abod, sod)
# showed no meaningful F1 gain from tuning and retain PyOD defaults.
#
# Source: checkpoints/phase1_clean_sensitivity.csv in v16_anomaly_diag/
# Promotion rule: F1 gain ≥ 0.03 OR FNR drop ≥ 0.05 over default.
# ─────────────────────────────────────────────────────────────────────────
TUNED_PARAMS = {
    'pca': {'n_components': 0.90},              # PCA-1 winner — ΔF1 = +0.1986
    'svm': {'nu': 0.01, 'gamma': 'scale'},      # SVM-1 winner — ΔF1 = +0.0585
}

total = len(CONFIG['MODELS']) * len(CONFIG['SEEDS']) * (1 + len(CONFIG['ATTACKS']) * len(CONFIG['POISON_RATES']))
print('Configuration loaded — PAPER RUN.')
print(f'  Models (12):   {CONFIG["MODELS"]}')
print(f'  Seeds:         {CONFIG["SEEDS"]}')
print(f'  Poison rates:  {CONFIG["POISON_RATES"]}')
print(f'  Attacks:       {CONFIG["ATTACKS"]}')
print(f'  Tuned models:  PCA={TUNED_PARAMS["pca"]}, SVM={TUNED_PARAMS["svm"]}')
print(f'  Total runs:    {total} (36 baselines + 432 attack runs)')


In [ ]:
# 1.4 Paths — Narval edition (env-var driven, no Drive mount)
import os

# Data must already live under $SWAT_DATA_DIR (default: $SCRATCH/swat_data).
# Outputs go under $SWAT_OUTPUT_DIR (default: $SCRATCH/swat_paper_run).
def _need(name):
    v = os.environ.get(name)
    if not v:
        raise RuntimeError(
            f"Environment variable {name} is not set. "
            "Export it before launching the notebook (e.g. in your .bashrc or "
            "`export {name}=$SCRATCH/...`).".format(name=name)
        )
    return v

SCRATCH = os.environ.get("SCRATCH") or _need("SCRATCH")
DATA_DIR    = os.environ.get("SWAT_DATA_DIR")   or os.path.join(SCRATCH, "swat_data")
OUTPUT_DIR  = os.environ.get("SWAT_OUTPUT_DIR") or os.path.join(SCRATCH, "swat_paper_run")
NORMAL_FILE = os.path.join(DATA_DIR, "normal.csv")
ATTACK_FILE = os.path.join(DATA_DIR, "attack.csv")

# Hard fail early rather than silently auto-discovering somewhere wrong.
for p in (NORMAL_FILE, ATTACK_FILE):
    if not os.path.exists(p):
        raise FileNotFoundError(
            f"Missing {p}. Upload the SWaT CSVs to {DATA_DIR} before running."
        )

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "checkpoints"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "figures"),     exist_ok=True)

# Kept for backwards compatibility with cells that reference it (we don't
# auto-discover, so the list is symbolic only).
IN_COLAB = False
MANUAL_DATA_DIR = None
MANUAL_OUTPUT_DIR = None
CANDIDATE_DIRS = [DATA_DIR]

print(f"DATA_DIR:    {DATA_DIR}")
print(f"NORMAL_FILE: {NORMAL_FILE} ({os.path.getsize(NORMAL_FILE):,} bytes)")
print(f"ATTACK_FILE: {ATTACK_FILE} ({os.path.getsize(ATTACK_FILE):,} bytes)")
print(f"OUTPUT_DIR:  {OUTPUT_DIR}")


In [ ]:
# 1.5 Reproducibility + robust CSV writer  (Narval edition — migration disabled)
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def safe_to_csv(df, path, **kwargs):
    tmp = path + ".tmp"
    df.to_csv(tmp, **kwargs)
    os.replace(tmp, path)

def migrate_checkpoints():
    # Disabled on Narval by design — this run is the single source of truth
    # for the paper. To re-enable, restore the body from the original notebook.
    print("Checkpoint migration: DISABLED (clean-slate Narval run).")

migrate_checkpoints()


---
## Pre-flight check

Review the output of the next cell carefully before proceeding. It prints:

- the resolved `OUTPUT_DIR` where all results will be written
- the exact `TUNED_PARAMS` dict being used
- whether any prior checkpoint files will be resumed (fresh folder = from-scratch run)

If `OUTPUT_DIR` already contains checkpoint CSVs, the runners will resume
from them. To force a completely fresh run, either delete the folder or
change `MANUAL_OUTPUT_DIR` in Cell 1.4 to a new path.


In [ ]:
# 1.6 Pre-flight check — confirm paths and tuned params before training
print('=' * 72)
print('PAPER RUN — PRE-FLIGHT CHECK')
print('=' * 72)
print(f'DATA_DIR:    {DATA_DIR}')
print(f'OUTPUT_DIR:  {OUTPUT_DIR}')
print()
print('TUNED_PARAMS (from Phase 1 sensitivity study):')
for model, params in TUNED_PARAMS.items():
    print(f'  {model:<10} {params}')
other = [m for m in CONFIG['MODELS'] if m not in TUNED_PARAMS and m not in ('autoencoder', 'lstm_ae')]
print(f'  others ({len(other)}): PyOD defaults — {other}')
print()

# Check for prior checkpoints in OUTPUT_DIR
cp_dir = os.path.join(OUTPUT_DIR, 'checkpoints')
existing_cps = []
if os.path.isdir(cp_dir):
    for f in sorted(os.listdir(cp_dir)):
        full = os.path.join(cp_dir, f)
        if os.path.isfile(full):
            existing_cps.append((f, os.path.getsize(full)))

if existing_cps:
    print('Existing checkpoints found — runners will RESUME from these:')
    for f, size in existing_cps:
        print(f'  {f} ({size:,} bytes)')
    print('\nTo force a fresh run, delete this folder or set a new MANUAL_OUTPUT_DIR in Cell 1.4.')
else:
    print('No prior checkpoints — this will be a FROM-SCRATCH paper run.')

print()
print('Grid plan: 12 models × 3 seeds × (1 clean + 3 attacks × 4 rates) = 468 runs.')
print('Budget estimate: 10-20 h on T4 GPU, 3-6 h on A100.')


---
## 2. Data Loading, Preprocessing, Splits
Same pipeline as R02. 44 features, ~450K samples, 12.14% attack ratio.


In [ ]:
# 2.1 Load CSVs
print('Loading data...')
df_normal = pd.read_csv(NORMAL_FILE)
df_attack = pd.read_csv(ATTACK_FILE)
print(f'  normal.csv: {len(df_normal):>8} rows')
print(f'  attack.csv: {len(df_attack):>8} rows')

In [ ]:
# 2.2 Preprocess — identical to R02 / v16_2 (proven)
def preprocess_swat(df_normal, df_attack):
    '''Clean SWaT data: drop label+timestamp, align columns, convert to numeric, drop constants.'''
    # 1) Identify label column in attack file
    label_col = None
    for col in df_attack.columns:
        if 'normal' in col.lower() or 'attack' in col.lower() or 'label' in col.lower():
            label_col = col; break
    print(f"Label column found: '{label_col}'")

    # 2) Create binary labels, drop the label-source column from features
    df_n = df_normal.copy()
    df_n['label'] = 0
    if label_col and label_col in df_n.columns:
        df_n = df_n.drop(columns=[label_col])

    df_a = df_attack.copy()
    if label_col:
        df_a['label'] = df_a[label_col].apply(lambda x: 1 if 'attack' in str(x).lower() else 0)
        df_a = df_a.drop(columns=[label_col])
    else:
        df_a['label'] = 1

    # 3) Drop any timestamp columns
    drop_cols = [c for c in df_n.columns if 'timestamp' in c.lower() or c.lower().strip() == 'time']
    df_n = df_n.drop(columns=[c for c in drop_cols if c in df_n.columns], errors='ignore')
    df_a = df_a.drop(columns=[c for c in drop_cols if c in df_a.columns], errors='ignore')

    # 4) Align columns by intersection
    common = sorted(list(set(df_n.columns) & set(df_a.columns)))
    df_n = df_n[common]; df_a = df_a[common]

    # 5) Combine
    df = pd.concat([df_n, df_a], ignore_index=True)

    # 6) Convert to numeric, drop NaN
    feats = [c for c in df.columns if c != 'label']
    for c in feats:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    n0 = len(df); df = df.dropna()
    print(f'Dropped {n0 - len(df)} rows with NaN values')

    # 7) Drop constant columns
    const = [c for c in feats if c in df.columns and df[c].nunique() <= 1]
    if const:
        print(f'Dropping {len(const)} constant columns: {const}')
        df = df.drop(columns=const)
    feats = [c for c in df.columns if c != 'label']
    print(f'\nFinal dataset: {len(df):,} samples, {len(feats)} features')
    print(f'Attack ratio: {df["label"].mean():.4f}')

    # 8) Rebuild cleaned per-file dataframes for the LSTM-AE contiguous split
    df_n_final = df_n.copy()
    df_a_final = df_a.copy()
    for c in feats:
        df_n_final[c] = pd.to_numeric(df_n_final[c], errors='coerce')
        df_a_final[c] = pd.to_numeric(df_a_final[c], errors='coerce')
    df_n_final = df_n_final.dropna()
    df_a_final = df_a_final.dropna()
    if const:
        df_n_final = df_n_final.drop(columns=[c for c in const if c in df_n_final.columns], errors='ignore')
        df_a_final = df_a_final.drop(columns=[c for c in const if c in df_a_final.columns], errors='ignore')
    print(f'\nFor LSTM-AE contiguous split:')
    print(f'  Normal block:  {len(df_n_final):,} samples')
    print(f'  Attack block:  {len(df_a_final):,} samples (attack ratio: {df_a_final["label"].mean():.4f})')
    return df, feats, df_n_final, df_a_final

df, FEATURE_COLS, df_normal_clean, df_attack_clean = preprocess_swat(df_normal, df_attack)

In [ ]:
# 2.3 Data splits — dual pipeline
# Pointwise models (10 PyOD + Autoencoder): random stratified split from the
#   concatenated normal+attack DataFrame.
# LSTM-AE:  contiguous split with FULL data usage — no samples discarded.

def create_splits(df, feats, test_size=0.15, val_size=0.15, seed=42):
    X = df[feats].values
    y = df['label'].values
    X_tmp, X_te, y_tmp, y_te = train_test_split(X, y, test_size=test_size, stratify=y, random_state=seed)
    v_ratio = val_size / (1 - test_size)
    X_tr, X_v, y_tr, y_v = train_test_split(X_tmp, y_tmp, test_size=v_ratio, stratify=y_tmp, random_state=seed)
    sc = MinMaxScaler()
    X_tr = sc.fit_transform(X_tr); X_v = sc.transform(X_v); X_te = sc.transform(X_te)
    X_tr_normal = X_tr[y_tr == 0]
    print(f'[Pointwise] Train: {len(X_tr):,} ({len(X_tr_normal):,} normal), Val: {len(X_v):,}, Test: {len(X_te):,}')
    return X_tr, X_tr_normal, X_v, X_te, y_tr, y_v, y_te, sc


def create_lstm_ae_splits(df_n, df_a, feats, seed=42):
    """
    FULL-DATA LSTM-AE split — uses 100% of normal.csv and 100% of attack.csv.

      train      = first 70% of normal.csv         (contiguous, LSTM reconstruction training)
      val-norm   = next 15% of normal.csv          (contiguous, early-stopping loss)
      val-mixed  = first 50% of attack.csv         (mixed labels, F1-optimal threshold calibration)
      test       = last 15% of normal.csv  +  last 50% of attack.csv
                   └─ held-out normal tail mixed with attack session's second half.
                      Forces the detector to flag attack windows WITHOUT flagging
                      temporally held-out normal windows → genuine evaluation.

    Note: 19 sliding windows at the normal-to-attack boundary span both sources.
    Their labels come from _create_sequence_labels (any timestep = attack → label 1),
    so they are counted correctly. 19 boundary windows out of ~150K is <0.02%.
    """
    Xn = df_n[feats].values
    Xa = df_a[feats].values
    ya = df_a['label'].values

    n = len(Xn)
    i_tr  = int(n * 0.70)
    i_vn  = int(n * 0.85)
    tr     = Xn[:i_tr]                       # train  (normal-only, 70%)
    vn     = Xn[i_tr:i_vn]                   # val-norm (normal-only, 15%)
    n_tail = Xn[i_vn:]                       # held-out normal tail (15%)

    mid   = len(Xa) // 2
    vm    = Xa[:mid];    yvm  = ya[:mid]     # val-mixed (attack.csv first half)
    a_te  = Xa[mid:];    y_ate = ya[mid:]    # attack.csv second half

    # Test = held-out normal tail + attack.csv second half
    te   = np.vstack([n_tail, a_te])
    yte  = np.concatenate([np.zeros(len(n_tail), dtype=int), y_ate])

    sc = MinMaxScaler().fit(tr)
    tr = sc.transform(tr); vn = sc.transform(vn)
    vm = sc.transform(vm); te = sc.transform(te)

    print(f'[LSTM-AE] Train (normal):      {len(tr):>7,}  (first 70% of normal.csv)')
    print(f'[LSTM-AE] Val-norm:            {len(vn):>7,}  (next 15% of normal.csv — early stopping)')
    print(f'[LSTM-AE] Val-mixed:           {len(vm):>7,}  (first half of attack.csv, attack ratio {yvm.mean():.4f})')
    print(f'[LSTM-AE] Test:                {len(te):>7,}  (held-out normal tail + attack.csv second half, attack ratio {yte.mean():.4f})')
    print(f'[LSTM-AE]   normal tail:       {len(n_tail):>7,}')
    print(f'[LSTM-AE]   attack half:       {len(a_te):>7,}  (attack ratio {y_ate.mean():.4f})')
    return tr, vn, vm, yvm, te, yte, sc


def subsample_if_heavy(X, model_name, seed=42):
    if model_name in CONFIG['HEAVY_MODELS'] and len(X) > CONFIG['MAX_TRAIN_SAMPLES']:
        rng = np.random.RandomState(seed)
        idx = rng.choice(len(X), CONFIG['MAX_TRAIN_SAMPLES'], replace=False)
        print(f'    [{model_name}] Subsampled: {len(X):,} -> {CONFIG["MAX_TRAIN_SAMPLES"]:,}')
        return X[idx]
    return X


# Sanity check on pointwise split
_ = create_splits(df, FEATURE_COLS, CONFIG['TEST_SIZE'], CONFIG['VAL_SIZE'], 42)


---
## 3. Model Definitions (12 anomaly detectors, tuned where relevant)


In [ ]:
# 3.1 Evaluation + F1-optimal threshold search (two-pass)
def evaluate(y_true, y_pred, y_scores=None):
    r = {
        'accuracy':  accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall':    recall_score(y_true, y_pred, zero_division=0),
        'f1':        f1_score(y_true, y_pred, zero_division=0),
    }
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    r['tp']=int(tp); r['tn']=int(tn); r['fp']=int(fp); r['fn']=int(fn)
    r['fnr'] = fn/(fn+tp) if (fn+tp) > 0 else 0
    r['fpr'] = fp/(fp+tn) if (fp+tn) > 0 else 0
    if y_scores is not None:
        try: r['roc_auc'] = roc_auc_score(y_true, y_scores)
        except: r['roc_auc'] = 0.5
        try: r['pr_auc']  = average_precision_score(y_true, y_scores)
        except: r['pr_auc']  = 0.0
    return r

def find_optimal_threshold(y_val, scores_val):
    '''Two-pass: percentile coarse scan + unique score-value fine search.'''
    if len(np.unique(y_val)) < 2:
        return float(np.median(scores_val))
    # coarse
    best_f1 = -1; best_t = float(np.median(scores_val))
    for q in np.linspace(1, 99, 99):
        t = float(np.percentile(scores_val, q))
        f1 = f1_score(y_val, (scores_val >= t).astype(int), zero_division=0)
        if f1 > best_f1: best_f1, best_t = f1, t
    # fine — unique score values near coarse best
    uniq = np.unique(scores_val)
    lo, hi = np.percentile(scores_val, max(0, 1)), np.percentile(scores_val, min(100, 99))
    fine = uniq[(uniq >= lo) & (uniq <= hi)]
    if len(fine) > 200:
        fine = fine[::len(fine)//200]
    for t in fine:
        f1 = f1_score(y_val, (scores_val >= t).astype(int), zero_division=0)
        if f1 > best_f1: best_f1, best_t = f1, float(t)
    return best_t

In [ ]:
# 3.2 PyOD wrapper — anomaly detector (consumes TUNED_PARAMS)
class AnomalyDetector:
    """
    Thin wrapper around PyOD detectors with F1-optimal threshold calibration.

    Reads hyperparameters from the global `TUNED_PARAMS` dict. Defaults from
    PyOD are used for any model not present in that dict.
    """

    def __init__(self, model_name, pyod_model=None, contamination=0.05, seed=42):
        self.model_name = model_name
        self.contamination = contamination
        self.seed = seed
        self.threshold = None
        self.model = pyod_model if pyod_model is not None else self._build_default()

    def _build_default(self):
        c = self.contamination
        tp = TUNED_PARAMS.get(self.model_name, {})
        if   self.model_name == 'iforest':   return IForest(contamination=c, random_state=self.seed)
        elif self.model_name == 'svm':       return OCSVM(contamination=c,
                                                          nu=tp.get('nu', 0.05),
                                                          gamma=tp.get('gamma', 'scale'))
        elif self.model_name == 'lof':       return LOF(contamination=c, novelty=True)
        elif self.model_name == 'cluster':   return CBLOF(contamination=c, random_state=self.seed)
        elif self.model_name == 'knn':       return KNN(contamination=c)
        elif self.model_name == 'histogram': return HBOS(contamination=c)
        elif self.model_name == 'pca':
            # If n_components is a float variance ratio, defer to train() which
            # uses sklearn.PCA on the actual training data to derive an int count.
            # If it's already an int, pass it through directly.
            ncomp = tp.get('n_components')
            if isinstance(ncomp, int):
                return PCA_AD(contamination=c, n_components=ncomp, random_state=self.seed)
            return PCA_AD(contamination=c, random_state=self.seed)   # placeholder, replaced in train()
        elif self.model_name == 'mcd':       return MCD(contamination=c, random_state=self.seed)
        elif self.model_name == 'abod':      return ABOD(contamination=c, method='fast', n_neighbors=10)
        elif self.model_name == 'sod':       return SOD(contamination=c, n_neighbors=20, ref_set=10)
        raise ValueError(f'Unknown model: {self.model_name}')

    def train(self, X_train_normal, _unused, X_val, y_val):
        # Resolve PCA variance-ratio → integer component count at fit time
        if self.model_name == 'pca':
            ncomp = TUNED_PARAMS.get('pca', {}).get('n_components')
            if isinstance(ncomp, float):
                tmp = SkPCA(n_components=ncomp, random_state=self.seed)
                tmp.fit(X_train_normal[:min(5000, len(X_train_normal))])
                self.model = PCA_AD(contamination=self.contamination,
                                    n_components=int(tmp.n_components_),
                                    random_state=self.seed)

        self.model.fit(X_train_normal)
        scores_val = self.model.decision_function(X_val)
        self.threshold = find_optimal_threshold(y_val, scores_val)

    def predict(self, X):
        return (self.model.decision_function(X) >= self.threshold).astype(int)

    def decision_scores(self, X):
        return self.model.decision_function(X)

print('AnomalyDetector wrapper loaded (TUNED_PARAMS wired in).')


In [ ]:
# 3.3 Autoencoder (V3) — pure anomaly, normal-only training
class AEModel(nn.Module):
    def __init__(self, input_dim, hidden_dims=[256,128,64], dropout=0.1):
        super().__init__()
        enc, prev = [], input_dim
        for h in hidden_dims:
            enc += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        self.encoder = nn.Sequential(*enc)
        dec, prev = [], hidden_dims[-1]
        for h in hidden_dims[-2::-1]:
            dec += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        dec += [nn.Linear(prev, input_dim)]
        self.decoder = nn.Sequential(*dec)
    def forward(self, x): return self.decoder(self.encoder(x))

class AutoencoderDetector:
    def __init__(self, input_dim, hidden_dims=None, seed=42):
        set_seed(seed)
        self.model = AEModel(input_dim,
                             hidden_dims=hidden_dims or CONFIG['AE_HIDDEN_DIMS'],
                             dropout=CONFIG['AE_DROPOUT']).to(DEVICE)
        self.threshold = None; self.seed = seed

    def train(self, X_train_normal, _unused, X_val, y_val):
        set_seed(self.seed)
        X_tr = torch.tensor(X_train_normal, dtype=torch.float32).to(DEVICE)
        ds = TensorDataset(X_tr)
        loader = DataLoader(ds, batch_size=1024, shuffle=True)

        opt = optim.Adam(self.model.parameters(), lr=CONFIG['AE_LR'], weight_decay=1e-5)
        sch = optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5)
        crit = nn.MSELoss()

        # split: 90% train, 10% val-normal for early stopping
        n = len(X_tr); vs = max(1, int(0.1*n))
        X_val_normal = X_tr[:vs]
        X_tr2 = X_tr[vs:]
        ds2 = TensorDataset(X_tr2); loader = DataLoader(ds2, batch_size=1024, shuffle=True)

        best_vl, bad = float('inf'), 0
        best_state = None
        for ep in range(CONFIG['AE_EPOCHS']):
            self.model.train()
            for (xb,) in loader:
                opt.zero_grad()
                out = self.model(xb); loss = crit(out, xb)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                opt.step()
            self.model.eval()
            with torch.no_grad():
                vl = crit(self.model(X_val_normal), X_val_normal).item()
            sch.step(vl)
            if vl < best_vl - 1e-5:
                best_vl, bad = vl, 0
                best_state = {k: v.clone() for k, v in self.model.state_dict().items()}
            else:
                bad += 1
                if bad >= CONFIG['AE_PATIENCE']: break
        if best_state is not None:
            self.model.load_state_dict(best_state)

        # Threshold on validation (mixed)
        self.model.eval()
        with torch.no_grad():
            Xv = torch.tensor(X_val, dtype=torch.float32).to(DEVICE)
            rec = self.model(Xv)
            err = ((rec - Xv)**2).mean(dim=1).cpu().numpy()
        self.threshold = find_optimal_threshold(y_val, err)

    def decision_scores(self, X):
        self.model.eval()
        with torch.no_grad():
            Xt = torch.tensor(X, dtype=torch.float32).to(DEVICE)
            rec = self.model(Xt)
            err = ((rec - Xt)**2).mean(dim=1).cpu().numpy()
        return err

    def predict(self, X):
        return (self.decision_scores(X) >= self.threshold).astype(int)

print('AutoencoderDetector loaded.')

In [ ]:
# 3.4 LSTM-Autoencoder — sequence reconstruction, contiguous normal training
class LSTMAEModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=1, dropout=0.2):
        super().__init__()
        self.encoder = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout if num_layers>1 else 0)
        self.decoder = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout if num_layers>1 else 0)
        self.out = nn.Linear(hidden_dim, input_dim)
    def forward(self, x):
        _, (h, c) = self.encoder(x)
        # feed reversed zeros with encoder state into decoder
        dec_in = torch.flip(x, dims=[1])
        dec_out, _ = self.decoder(dec_in, (h, c))
        rec = self.out(dec_out)
        return torch.flip(rec, dims=[1])

class LSTMAEDetector:
    def __init__(self, input_dim, seed=42):
        set_seed(seed)
        self.model = LSTMAEModel(input_dim,
                                 hidden_dim=CONFIG['LSTM_AE_HIDDEN'],
                                 dropout=CONFIG['LSTM_AE_DROPOUT']).to(DEVICE)
        self.window = CONFIG['LSTM_AE_WINDOW']
        self.threshold = None; self.seed = seed; self.input_dim = input_dim

    def _seq(self, X):
        X = np.asarray(X, dtype=np.float32)
        W = self.window
        if len(X) < W: return np.empty((0, W, X.shape[1]), dtype=np.float32)
        out = np.stack([X[i:i+W] for i in range(len(X) - W + 1)])
        return out

    def _create_sequence_labels(self, y):
        W = self.window
        y = np.asarray(y)
        if len(y) < W: return np.empty(0, dtype=int)
        return np.stack([1 if y[i:i+W].max() > 0 else 0 for i in range(len(y) - W + 1)])

    def train(self, X_train_normal, X_val_normal, X_val, y_val):
        set_seed(self.seed)
        Xs_tr = torch.tensor(self._seq(X_train_normal), dtype=torch.float32)
        Xs_vn = torch.tensor(self._seq(X_val_normal), dtype=torch.float32)
        loader = DataLoader(TensorDataset(Xs_tr), batch_size=512, shuffle=True)

        opt = optim.Adam(self.model.parameters(), lr=CONFIG['LSTM_AE_LR'], weight_decay=1e-5)
        sch = optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5)
        crit = nn.MSELoss()

        best_vl, bad, best_state = float('inf'), 0, None
        for ep in range(CONFIG['LSTM_AE_EPOCHS']):
            self.model.train()
            for (xb,) in loader:
                xb = xb.to(DEVICE); opt.zero_grad()
                rec = self.model(xb); loss = crit(rec, xb)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                opt.step()
            self.model.eval()
            with torch.no_grad():
                # val loss on contiguous normal
                errs = []
                for i in range(0, len(Xs_vn), 512):
                    b = Xs_vn[i:i+512].to(DEVICE)
                    errs.append(crit(self.model(b), b).item())
                vl = float(np.mean(errs)) if errs else float('inf')
            sch.step(vl)
            if vl < best_vl - 1e-5:
                best_vl, bad = vl, 0
                best_state = {k: v.clone() for k, v in self.model.state_dict().items()}
            else:
                bad += 1
                if bad >= CONFIG['LSTM_AE_PATIENCE']: break
        if best_state is not None: self.model.load_state_dict(best_state)

        # Threshold on mixed validation
        scores = self.decision_scores(X_val)
        y_seq  = self._create_sequence_labels(y_val)
        n = min(len(scores), len(y_seq))
        self.threshold = find_optimal_threshold(y_seq[:n], scores[:n])

    def decision_scores(self, X):
        self.model.eval()
        Xs = self._seq(X)
        if len(Xs) == 0: return np.empty(0)
        Xt = torch.tensor(Xs, dtype=torch.float32)
        errs = []
        with torch.no_grad():
            for i in range(0, len(Xt), 512):
                b = Xt[i:i+512].to(DEVICE)
                rec = self.model(b)
                e = ((rec - b)**2).mean(dim=(1,2)).cpu().numpy()
                errs.append(e)
        return np.concatenate(errs) if errs else np.empty(0)

    def predict(self, X):
        return (self.decision_scores(X) >= self.threshold).astype(int)

print('LSTMAEDetector loaded.')

In [ ]:
# 3.5 Unified model factory
PYOD_MODELS  = {'iforest','svm','lof','cluster','knn','histogram','pca','mcd','abod','sod'}
TORCH_MODELS = {'autoencoder','lstm_ae'}

def create_model(name, input_dim, seed=42):
    if name in PYOD_MODELS:
        return AnomalyDetector(name, contamination=CONFIG['CONTAMINATION'], seed=seed)
    if name == 'autoencoder':
        return AutoencoderDetector(input_dim, seed=seed)
    if name == 'lstm_ae':
        return LSTMAEDetector(input_dim, seed=seed)
    raise ValueError(name)

print('Model factory ready. PyOD:', sorted(PYOD_MODELS))
print('Torch:', sorted(TORCH_MODELS))

---
## 4. Poisoning Attacks (Unified Anomaly Paradigm)

All models train on normal-only → poisoning contaminates the normal training pool.

| Attack | Mechanism | R02 analog |
|---|---|---|
| `random_flip` | Inject random attack samples into normal pool | Algorithm 1 |
| `targeted_flip` | Inject attack samples closest to normal centroid | Algorithm 2 |
| `feature_noise` | Add Gaussian noise to normal training features | Algorithm 3 |


In [ ]:
# 4.1 Random injection
def random_injection(X_train_normal, X_train_full, y_train_full, rate, seed=42):
    rng = np.random.RandomState(seed)
    X_atk = X_train_full[y_train_full == 1]
    if len(X_atk) == 0: return X_train_normal.copy(), {'n_injected':0}
    k = min(int(rate * len(X_train_normal)), len(X_atk))
    idx = rng.choice(len(X_atk), k, replace=False)
    X_new = np.vstack([X_train_normal, X_atk[idx]])
    return X_new, {'n_injected': k, 'effective_contamination': k/len(X_new)}

# 4.2 Targeted injection
def targeted_injection(X_train_normal, X_train_full, y_train_full, rate, seed=42):
    X_atk = X_train_full[y_train_full == 1]
    if len(X_atk) == 0: return X_train_normal.copy(), {'n_injected':0}
    mu = X_train_normal.mean(axis=0)
    d = np.linalg.norm(X_atk - mu, axis=1)
    k = min(int(rate * len(X_train_normal)), len(X_atk))
    idx = np.argsort(d)[:k]
    X_new = np.vstack([X_train_normal, X_atk[idx]])
    return X_new, {'n_injected': k, 'effective_contamination': k/len(X_new)}

# 4.3 Feature noise
def feature_noise_injection(X_train_normal, X_train_full, y_train_full, rate, seed=42):
    rng = np.random.RandomState(seed)
    X_new = X_train_normal.copy()
    k = int(rate * len(X_new))
    idx = rng.choice(len(X_new), k, replace=False)
    std = X_new.std(axis=0)
    noise = rng.normal(0, CONFIG['NOISE_SIGMA']*std, size=X_new[idx].shape)
    X_new[idx] = np.clip(X_new[idx] + noise, 0, 1)
    return X_new, {'n_poisoned': k}

def apply_poison(X_train_normal, X_train_full, y_train_full, attack_name, rate, seed=42):
    if   attack_name == 'random_flip':   return random_injection(X_train_normal, X_train_full, y_train_full, rate, seed)
    elif attack_name == 'targeted_flip': return targeted_injection(X_train_normal, X_train_full, y_train_full, rate, seed)
    elif attack_name == 'feature_noise': return feature_noise_injection(X_train_normal, X_train_full, y_train_full, rate, seed)
    raise ValueError(attack_name)

print('Poisoning factory ready.')

---
## 5. Clean Baselines (12 models × 3 seeds = 36 runs)


In [ ]:
# 5.1 Run clean baselines (checkpointed)
def run_clean_baselines():
    bl_file = os.path.join(OUTPUT_DIR, 'checkpoints', 'clean_baselines.csv')
    rows = []
    done = set()
    if os.path.exists(bl_file):
        try:
            ex = pd.read_csv(bl_file); rows = ex.to_dict('records')
            for r in rows: done.add((r['model'], int(r['seed'])))
            print(f'Resuming: {len(done)} baselines already done.')
        except: pass

    input_dim = len(FEATURE_COLS)
    for seed in CONFIG['SEEDS']:
        print(f'\n--- Seed {seed} ---')
        set_seed(seed)
        X_tr, X_n, X_v, X_te, y_tr, y_v, y_te, _ = create_splits(df, FEATURE_COLS, CONFIG['TEST_SIZE'], CONFIG['VAL_SIZE'], seed)
        lstm_sp = create_lstm_ae_splits(df_normal_clean, df_attack_clean, FEATURE_COLS, seed)
        X_sq_tr, X_sq_vn, X_sq_vm, y_sq_vm, X_sq_te, y_sq_te, _ = lstm_sp

        for m in CONFIG['MODELS']:
            if (m, seed) in done:
                print(f'  skip {m}'); continue
            t0 = time.time()
            try:
                det = create_model(m, input_dim, seed=seed)
                if m == 'lstm_ae':
                    det.train(X_sq_tr, X_sq_vn, X_sq_vm, y_sq_vm)
                    y_scores = det.decision_scores(X_sq_te)
                    y_eval   = det._create_sequence_labels(y_sq_te)
                    n = min(len(y_scores), len(y_eval))
                    y_pred = (y_scores[:n] >= det.threshold).astype(int)
                    met = evaluate(y_eval[:n], y_pred, y_scores[:n])
                else:
                    X_train_use = subsample_if_heavy(X_n, m, seed)
                    det.train(X_train_use, None, X_v, y_v)
                    y_pred   = det.predict(X_te)
                    y_scores = det.decision_scores(X_te)
                    met = evaluate(y_te, y_pred, y_scores)
                met.update({'model': m, 'seed': seed, 'time': time.time()-t0,
                            'train_normal_size': len(X_sq_tr) if m == 'lstm_ae' else len(X_train_use),
                            'split_type': 'contiguous_normal' if m == 'lstm_ae' else 'random'})
                rows.append(met)
                print(f'  {m:<12} F1={met["f1"]:.4f} FNR={met["fnr"]:.4f} t={time.time()-t0:.1f}s')
                safe_to_csv(pd.DataFrame(rows), bl_file, index=False)
                del det
                if torch.cuda.is_available(): torch.cuda.empty_cache()
                gc.collect()
            except Exception as e:
                print(f'  {m:<12} FAILED: {e}')
                rows.append({'model': m, 'seed': seed, 'error': str(e), 'f1': 0, 'fnr': 1})
                safe_to_csv(pd.DataFrame(rows), bl_file, index=False)
    return pd.DataFrame(rows)

clean_df = run_clean_baselines()

---
### 5.1b LSTM-AE score-distribution diagnostic

Validates that the LSTM-AE's reconstruction error genuinely separates
normal from attack windows. Reports AUC, average precision, the
separation gap, and saves a two-panel histogram
(`figures/lstm_ae_score_distribution.png`) for the paper appendix.


In [ ]:
# 5.1b LSTM-AE diagnostic — verify separation is real (runs once at seed=42)
from sklearn.metrics import roc_auc_score, average_precision_score

print('=' * 72)
print('LSTM-AE SCORE-DISTRIBUTION DIAGNOSTIC  (seed=42)')
print('=' * 72)

seed = 42
set_seed(seed)
_ = create_splits(df, FEATURE_COLS, CONFIG['TEST_SIZE'], CONFIG['VAL_SIZE'], seed)
lstm_sp = create_lstm_ae_splits(df_normal_clean, df_attack_clean, FEATURE_COLS, seed)
X_sq_tr, X_sq_vn, X_sq_vm, y_sq_vm, X_sq_te, y_sq_te, _ = lstm_sp

det = create_model('lstm_ae', len(FEATURE_COLS), seed=seed)
det.train(X_sq_tr, X_sq_vn, X_sq_vm, y_sq_vm)

scores = det.decision_scores(X_sq_te)
y_win  = det._create_sequence_labels(y_sq_te)
n      = min(len(scores), len(y_win))
scores, y_win = scores[:n], y_win[:n]

n_norm, n_atk = (y_win == 0).sum(), (y_win == 1).sum()
s_norm, s_atk = scores[y_win == 0], scores[y_win == 1]
gap = s_atk.min() - s_norm.max() if n_atk and n_norm else float('nan')

auc = roc_auc_score(y_win, scores)
ap  = average_precision_score(y_win, scores)

print(f'Total test windows:  {n:,}')
print(f'  Normal windows:    {n_norm:,}')
print(f'  Attack windows:    {n_atk:,}')
print(f'  Attack ratio:      {y_win.mean():.4f}')
print(f'Threshold (F1-opt):  {det.threshold:.6e}')
print()
print('Reconstruction error — per-class statistics:')
print(f'  Normal:  min={s_norm.min():.3e}  max={s_norm.max():.3e}  mean={s_norm.mean():.3e}')
print(f'  Attack:  min={s_atk.min():.3e}  max={s_atk.max():.3e}  mean={s_atk.mean():.3e}')
print(f'  Min-attack minus max-normal gap:  {gap:.3e}   '
      f'({"SEPARABLE" if gap > 0 else "OVERLAPPING"})')
print()
print(f'AUC:                 {auc:.6f}')
print(f'Average precision:   {ap:.6f}')

# Two-panel histogram — linear + log scale
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))

ax[0].hist(s_norm, bins=60, alpha=0.7, label=f'Normal (n={n_norm:,})', color='#4C72B0')
ax[0].hist(s_atk,  bins=60, alpha=0.7, label=f'Attack (n={n_atk:,})', color='#C44E52')
ax[0].axvline(det.threshold, color='black', linestyle='--',
              label=f'threshold={det.threshold:.2e}')
ax[0].set_xlabel('Reconstruction error'); ax[0].set_ylabel('Window count')
ax[0].set_title('LSTM-AE score distribution (linear)')
ax[0].legend(loc='upper right'); ax[0].grid(alpha=0.3)

ax[1].hist(np.log10(s_norm + 1e-12), bins=60, alpha=0.7, label='Normal', color='#4C72B0')
ax[1].hist(np.log10(s_atk  + 1e-12), bins=60, alpha=0.7, label='Attack', color='#C44E52')
ax[1].axvline(np.log10(det.threshold + 1e-12), color='black', linestyle='--',
              label=f'threshold')
ax[1].set_xlabel('log10(reconstruction error)'); ax[1].set_ylabel('Window count')
ax[1].set_title('LSTM-AE score distribution (log scale)')
ax[1].legend(loc='upper right'); ax[1].grid(alpha=0.3)

plt.suptitle(
    f'LSTM-AE clean-baseline diagnostic — AUC={auc:.4f}, AP={ap:.4f}',
    fontsize=12, fontweight='bold')
plt.tight_layout()
out_png = os.path.join(OUTPUT_DIR, 'figures', 'lstm_ae_score_distribution.png')
safe_savefig = lambda p, **kw: (os.makedirs(os.path.dirname(p), exist_ok=True), plt.savefig(p, **kw))
safe_savefig(out_png, dpi=200, bbox_inches='tight')
plt.show()
print(f'\nSaved: {out_png}')

# Clean up GPU memory before the grid starts
del det
if torch.cuda.is_available(): torch.cuda.empty_cache()
gc.collect()


In [ ]:
# 5.2 Clean baseline summary (Table T4)
print('\n' + '='*72)
print('TABLE T4 — Clean Baseline Performance (mean \u00b1 std over 3 seeds)')
print('='*72)
print(f'{"Model":<14} {"F1":>14} {"Recall":>14} {"Precision":>14} {"FNR":>14} {"Time(s)":>10}')
print('─'*72)
for m in CONFIG['MODELS']:
    # Initialize filter to select rows matching the model
    model_filter = clean_df['model'] == m
    # If 'error' column exists, also filter for rows where 'error' is NaN (i.e., no error)
    if 'error' in clean_df.columns:
        model_filter = model_filter & clean_df['error'].isna()
    sub = clean_df[model_filter]

    if len(sub) == 0: continue
    def ms(k): return f'{sub[k].mean():.4f} \u00b1 {sub[k].std():.4f}'

    # Handle column name discrepancy between checkpoints
    time_col = 'time' if 'time' in sub.columns else 'train_time'
    time_val = sub[time_col].mean() if time_col in sub.columns else 0.0

    print(f'{m:<14} {ms("f1"):>14} {ms("recall"):>14} {ms("precision"):>14} {ms("fnr"):>14} {time_val:>10.1f}')
clean_df.to_csv(os.path.join(OUTPUT_DIR, 'table_T4_clean_baselines.csv'), index=False)

---
## 6. Full Poisoning Grid (432 runs)

**Grid**: 12 models × 3 attacks × 4 rates × 3 seeds.
Checkpointed — safe to stop and resume.


In [ ]:
# 6.1 Single-experiment runner
def run_single(m, X_n, X_tr, y_tr, X_v, y_v, X_te, y_te, input_dim, attack, rate, seed,
               lstm_sp=None):
    t0 = time.time()
    if m == 'lstm_ae':
        X_sq_tr, X_sq_vn, X_sq_vm, y_sq_vm, X_sq_te, y_sq_te, _ = lstm_sp
        # Poison the contiguous-normal training block
        X_poisoned, info = apply_poison(X_sq_tr, X_tr, y_tr, attack, rate, seed)
        det = create_model('lstm_ae', input_dim, seed=seed)
        det.train(X_poisoned, X_sq_vn, X_sq_vm, y_sq_vm)
        y_scores = det.decision_scores(X_sq_te)
        y_eval   = det._create_sequence_labels(y_sq_te)
        n = min(len(y_scores), len(y_eval))
        y_pred = (y_scores[:n] >= det.threshold).astype(int)
        met = evaluate(y_eval[:n], y_pred, y_scores[:n])
    else:
        X_poisoned, info = apply_poison(X_n, X_tr, y_tr, attack, rate, seed)
        X_train_use = subsample_if_heavy(X_poisoned, m, seed)
        det = create_model(m, input_dim, seed=seed)
        det.train(X_train_use, None, X_v, y_v)
        y_pred   = det.predict(X_te)
        y_scores = det.decision_scores(X_te)
        met = evaluate(y_te, y_pred, y_scores)
    met.update({'model': m, 'attack': attack, 'poison_rate': rate, 'seed': seed,
                'time': time.time()-t0,
                'split_type': 'contiguous_normal' if m == 'lstm_ae' else 'random',
                'effective_contamination': info.get('effective_contamination', None),
                'threshold': float(det.threshold) if det.threshold is not None else None})
    del det
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    gc.collect()
    return met

print('Single-experiment runner loaded.')

> ⚠️ **Full grid — budget this cell carefully**
>
> The cell below runs **432 experiments** (12 models × 3 attacks × 4 rates
> × 3 seeds). On an A100 40GB this typically takes **3–5 hours**.
>
> Options:
> - Run here, cell-by-cell, inside a long `salloc` — good for the first end-to-end
>   run with full visibility.
> - Kill it anytime — checkpoints in `$SWAT_OUTPUT_DIR/checkpoints/attack_checkpoint.csv`
>   let you resume simply by rerunning this cell.
> - For the "serious" batch run, use the Slurm arrays instead:
>   `sbatch slurm/20_attack_random_flip.sh` etc. Those use per-combo JSONs
>   and parallelize across GPUs.


In [ ]:
# 6.2 Grid runner (checkpointed)
def run_attack_grid():
    att_file = os.path.join(OUTPUT_DIR, 'checkpoints', 'attack_checkpoint.csv')
    bl_file  = os.path.join(OUTPUT_DIR, 'checkpoints', 'clean_baselines.csv')

    rows, done = [], set()
    # Include clean baselines as attack='none'
    if os.path.exists(bl_file):
        try:
            bdf = pd.read_csv(bl_file)
            for _, r in bdf.iterrows():
                d = r.to_dict()
                d['attack'] = 'none'; d['poison_rate'] = 0
                rows.append(d)
                done.add((d['model'], 'none', 0.0, int(d['seed'])))
        except: pass
    if os.path.exists(att_file):
        try:
            adf = pd.read_csv(att_file)
            for _, r in adf.iterrows():
                d = r.to_dict(); rows.append(d)
                done.add((d['model'], d['attack'], float(d['poison_rate']), int(d['seed'])))
            print(f'Resuming: {len(done)} runs already done.')
        except: pass

    input_dim = len(FEATURE_COLS)
    total = len(CONFIG['MODELS']) * len(CONFIG['SEEDS']) * (1 + len(CONFIG['ATTACKS']) * len(CONFIG['POISON_RATES']))
    i = len(done)

    for seed in CONFIG['SEEDS']:
        print(f'\n{"="*56}\nSEED {seed}\n{"="*56}')
        set_seed(seed)
        X_tr, X_n, X_v, X_te, y_tr, y_v, y_te, _ = create_splits(df, FEATURE_COLS, CONFIG['TEST_SIZE'], CONFIG['VAL_SIZE'], seed)
        lstm_sp = create_lstm_ae_splits(df_normal_clean, df_attack_clean, FEATURE_COLS, seed)

        for m in CONFIG['MODELS']:
            for attack in CONFIG['ATTACKS']:
                for rate in CONFIG['POISON_RATES']:
                    key = (m, attack, rate, seed)
                    if key in done: continue
                    i += 1
                    print(f'[{i}/{total}] {m} | {attack} r={rate} | seed={seed}')
                    try:
                        met = run_single(m, X_n, X_tr, y_tr, X_v, y_v, X_te, y_te,
                                         input_dim, attack, rate, seed, lstm_sp=lstm_sp)
                        rows.append(met)
                        print(f'   -> F1={met["f1"]:.4f} FNR={met["fnr"]:.4f}')
                    except Exception as e:
                        print(f'   -> FAILED: {e}')
                        rows.append({'model':m,'attack':attack,'poison_rate':rate,'seed':seed,
                                     'error':str(e),'f1':0,'fnr':1})
                    # Save only attack rows (not baselines again)
                    att_only = [r for r in rows if r.get('attack') not in (None, 'none')]
                    safe_to_csv(pd.DataFrame(att_only), att_file, index=False)
    return pd.DataFrame(rows)

results_df = run_attack_grid()
print(f'\nGrid complete: {len(results_df)} total rows (incl. baselines).')
results_df.to_csv(os.path.join(OUTPUT_DIR, 'all_results.csv'), index=False)

---
## 7. Results Tables


In [ ]:
# 7.1 Load for analysis
results_df = pd.read_csv(os.path.join(OUTPUT_DIR, 'all_results.csv'))
if 'error' in results_df.columns:
    results_df = results_df[results_df['error'].isna()].copy()
clean_df    = results_df[results_df['attack'] == 'none'].copy()
poisoned_df = results_df[results_df['attack'] != 'none'].copy()
print(f'Clean rows: {len(clean_df)} | Poisoned rows: {len(poisoned_df)}')

In [ ]:
# 7.2 Table T5 — Poisoning impact (F1 + FNR, mean over 3 seeds)
print('\n' + '='*90)
print('TABLE T5 — Poisoning Impact (F1 / FNR, mean over 3 seeds)')
print('='*90)
rows = []
for m in CONFIG['MODELS']:
    clean_f1  = clean_df[clean_df['model']==m]['f1'].mean()
    clean_fnr = clean_df[clean_df['model']==m]['fnr'].mean()
    row = {'model': m, 'clean_f1': clean_f1, 'clean_fnr': clean_fnr}
    for a in CONFIG['ATTACKS']:
        for r in CONFIG['POISON_RATES']:
            s = poisoned_df[(poisoned_df['model']==m) & (poisoned_df['attack']==a) & (poisoned_df['poison_rate']==r)]
            row[f'{a}_{r}_f1']  = s['f1'].mean()  if len(s) else np.nan
            row[f'{a}_{r}_fnr'] = s['fnr'].mean() if len(s) else np.nan
    rows.append(row)
T5 = pd.DataFrame(rows)
T5.to_csv(os.path.join(OUTPUT_DIR, 'table_T5_poisoning_impact.csv'), index=False)

# Compact printout
for m in CONFIG['MODELS']:
    r = T5[T5['model']==m].iloc[0]
    print(f'\n{m.upper():<14}  clean F1={r["clean_f1"]:.4f}  clean FNR={r["clean_fnr"]:.4f}')
    for a in CONFIG['ATTACKS']:
        f1s  = [r.get(f"{a}_{rt}_f1")  for rt in CONFIG['POISON_RATES']]
        fnrs = [r.get(f"{a}_{rt}_fnr") for rt in CONFIG['POISON_RATES']]
        f1_str  = '  '.join(f'{v:.3f}' if pd.notna(v) else ' NA  ' for v in f1s)
        fnr_str = '  '.join(f'{v:.3f}' if pd.notna(v) else ' NA  ' for v in fnrs)
        print(f'  {a:<14} F1  @ {CONFIG["POISON_RATES"]} = {f1_str}')
        print(f'  {"":<14} FNR @ {CONFIG["POISON_RATES"]} = {fnr_str}')

---
## 8. Publication Figures


In [ ]:
# 8.1 Figure F3 — Robustness curves (F1 vs poison rate, one subplot per attack)
fig, axes = plt.subplots(1, len(CONFIG['ATTACKS']), figsize=(6*len(CONFIG['ATTACKS']), 5), sharey=True)
colors = plt.cm.tab20(np.linspace(0, 1, len(CONFIG['MODELS'])))
for ax, a in zip(axes, CONFIG['ATTACKS']):
    for i, m in enumerate(CONFIG['MODELS']):
        ys, xs = [], []
        clean = clean_df[clean_df['model']==m]['f1'].mean()
        xs.append(0); ys.append(clean)
        for r in CONFIG['POISON_RATES']:
            s = poisoned_df[(poisoned_df['model']==m) & (poisoned_df['attack']==a) & (poisoned_df['poison_rate']==r)]
            if len(s):
                xs.append(r); ys.append(s['f1'].mean())
        ax.plot(xs, ys, '-o', color=colors[i], label=m, markersize=5, linewidth=1.5)
    ax.set_title(a.replace('_', ' ').title(), fontsize=12)
    ax.set_xlabel('Poison Rate'); ax.grid(alpha=0.3)
    ax.set_xticks([0] + CONFIG['POISON_RATES'])
axes[0].set_ylabel('F1-score')
axes[-1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8)
plt.suptitle('F3: Robustness Curves', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'F3_robustness_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 8.2 Figure F4 — F1 degradation heatmap (Δ from clean baseline)
fig, axes = plt.subplots(1, len(CONFIG['ATTACKS']), figsize=(6*len(CONFIG['ATTACKS']), max(5, 0.5*len(CONFIG['MODELS']))))
for ax, a in zip(axes, CONFIG['ATTACKS']):
    mat = np.full((len(CONFIG['MODELS']), len(CONFIG['POISON_RATES'])), np.nan)
    for i, m in enumerate(CONFIG['MODELS']):
        clean = clean_df[clean_df['model']==m]['f1'].mean()
        for j, r in enumerate(CONFIG['POISON_RATES']):
            s = poisoned_df[(poisoned_df['model']==m) & (poisoned_df['attack']==a) & (poisoned_df['poison_rate']==r)]
            if len(s): mat[i,j] = s['f1'].mean() - clean
    sns.heatmap(mat, ax=ax, annot=True, fmt='+.3f', cmap='RdBu_r', center=0, vmin=-0.5, vmax=0.1,
                xticklabels=CONFIG['POISON_RATES'], yticklabels=CONFIG['MODELS'], cbar_kws={'label':'ΔF1 from clean'})
    ax.set_title(a.replace('_',' ').title())
    ax.set_xlabel('Poison Rate')
plt.suptitle('F4: F1 Degradation Heatmap (ΔF1 from clean baseline)', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'F4_degradation_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 8.3 Figure F5 — FNR safety analysis
fig, axes = plt.subplots(1, len(CONFIG['ATTACKS']), figsize=(6*len(CONFIG['ATTACKS']), 5), sharey=True)
for ax, a in zip(axes, CONFIG['ATTACKS']):
    for i, m in enumerate(CONFIG['MODELS']):
        xs, ys = [0], [clean_df[clean_df['model']==m]['fnr'].mean()]
        for r in CONFIG['POISON_RATES']:
            s = poisoned_df[(poisoned_df['model']==m) & (poisoned_df['attack']==a) & (poisoned_df['poison_rate']==r)]
            if len(s):
                xs.append(r); ys.append(s['fnr'].mean())
        ax.plot(xs, ys, '-o', color=colors[i], label=m, markersize=5, linewidth=1.5)
    ax.axhline(0.10, color='red', linestyle='--', alpha=0.6, label='Safety threshold (FNR=10%)')
    ax.set_title(a.replace('_',' ').title()); ax.set_xlabel('Poison Rate'); ax.grid(alpha=0.3)
axes[0].set_ylabel('False Negative Rate (missed attacks)')
axes[-1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8)
plt.suptitle('F5: FNR Safety Analysis (lower is safer)', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'F5_fnr_safety.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 8.4 Figure F6 — Seed variance (±σ bands)
fig, axes = plt.subplots(1, len(CONFIG['ATTACKS']), figsize=(6*len(CONFIG['ATTACKS']), 5), sharey=True)
for ax, a in zip(axes, CONFIG['ATTACKS']):
    for i, m in enumerate(CONFIG['MODELS']):
        rates = [0] + CONFIG['POISON_RATES']
        means, stds = [], []
        clean_sub = clean_df[clean_df['model']==m]['f1']
        means.append(clean_sub.mean()); stds.append(clean_sub.std())
        for r in CONFIG['POISON_RATES']:
            s = poisoned_df[(poisoned_df['model']==m) & (poisoned_df['attack']==a) & (poisoned_df['poison_rate']==r)]
            means.append(s['f1'].mean() if len(s) else np.nan)
            stds.append(s['f1'].std() if len(s) else 0)
        means, stds = np.array(means), np.array(stds)
        ax.plot(rates, means, '-', color=colors[i], label=m)
        ax.fill_between(rates, means-stds, means+stds, alpha=0.15, color=colors[i])
    ax.set_title(a.replace('_',' ').title()); ax.set_xlabel('Poison Rate'); ax.grid(alpha=0.3)
axes[0].set_ylabel('F1 (mean ± std over seeds)')
axes[-1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8)
plt.suptitle('F6: Seed Variance', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'F6_seed_variance.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 8.5 Figure F7 — Per-model comparison (one subplot per model, all 3 attacks)
n_models = len(CONFIG['MODELS'])
n_cols = 4; n_rows = (n_models + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 3.5*n_rows))
axes = axes.flatten()
attack_colors = {'random_flip':'tab:blue','targeted_flip':'tab:red','feature_noise':'tab:green'}
for idx, m in enumerate(CONFIG['MODELS']):
    ax = axes[idx]
    clean = clean_df[clean_df['model']==m]['f1'].mean()
    for a in CONFIG['ATTACKS']:
        xs = [0]; ys = [clean]
        for r in CONFIG['POISON_RATES']:
            s = poisoned_df[(poisoned_df['model']==m) & (poisoned_df['attack']==a) & (poisoned_df['poison_rate']==r)]
            if len(s):
                xs.append(r); ys.append(s['f1'].mean())
        ax.plot(xs, ys, '-o', color=attack_colors[a], label=a, markersize=4)
    ax.set_title(m); ax.set_xlabel('Rate'); ax.set_ylabel('F1'); ax.grid(alpha=0.3); ax.legend(fontsize=7)
for i in range(n_models, len(axes)): axes[i].set_visible(False)
plt.suptitle('F7: Per-Model Comparison Across Attacks', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'F7_per_model.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 8.6 Figure F8 — Precision vs recall trade-off
fig, axes = plt.subplots(1, len(CONFIG['ATTACKS']), figsize=(6*len(CONFIG['ATTACKS']), 5))
for ax, a in zip(axes, CONFIG['ATTACKS']):
    for i, m in enumerate(CONFIG['MODELS']):
        xs, ys = [], []
        clean_sub = clean_df[clean_df['model']==m]
        if len(clean_sub):
            xs.append(clean_sub['recall'].mean()); ys.append(clean_sub['precision'].mean())
        for r in CONFIG['POISON_RATES']:
            s = poisoned_df[(poisoned_df['model']==m) & (poisoned_df['attack']==a) & (poisoned_df['poison_rate']==r)]
            if len(s):
                xs.append(s['recall'].mean()); ys.append(s['precision'].mean())
        ax.plot(xs, ys, '-o', color=colors[i], label=m, markersize=4, alpha=0.7)
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision'); ax.set_title(a.replace('_',' ').title())
    ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02); ax.grid(alpha=0.3)
axes[-1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8)
plt.suptitle('F8: Precision–Recall Trade-off (trajectory from clean → poisoned)', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'F8_precision_recall.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Diagnostic Analyses (appendix material)

Explains *why* certain models appear weak in the main grid. These analyses
run once at seed=42 and should be reported in the paper's diagnostic section
or appendix.


In [ ]:
# 9.1 Score-distribution analysis + top-k ranking — why are some models weak?
DIAG_MODELS = ['iforest','svm','lof','cluster','knn','histogram','pca','mcd','abod','sod']
seed = 42; set_seed(seed)
X_tr, X_n, X_v, X_te, y_tr, y_v, y_te, _ = create_splits(df, FEATURE_COLS, CONFIG['TEST_SIZE'], CONFIG['VAL_SIZE'], seed)

from sklearn.metrics import roc_auc_score, average_precision_score
rank_rows = []
fig, axes = plt.subplots(2, 5, figsize=(22, 8))
axes = axes.flatten()

for i, m in enumerate(DIAG_MODELS):
    det = create_model(m, len(FEATURE_COLS), seed=seed)
    X_use = subsample_if_heavy(X_n, m, seed)
    det.train(X_use, None, X_v, y_v)
    s = det.decision_scores(X_te)
    try:    roc = roc_auc_score(y_te, s)
    except: roc = 0.5
    try:    pr  = average_precision_score(y_te, s)
    except: pr  = 0.0
    rank_rows.append({'model': m, 'roc_auc': roc, 'pr_auc': pr, 'threshold': det.threshold})

    ax = axes[i]
    bins = np.linspace(s.min(), s.max(), 60)
    ax.hist(s[y_te==0], bins=bins, alpha=0.5, label='normal', color='blue', density=True)
    ax.hist(s[y_te==1], bins=bins, alpha=0.5, label='attack', color='red', density=True)
    ax.axvline(det.threshold, color='k', linestyle='--', label=f'thr={det.threshold:.3g}')
    ax.set_title(f'{m}\nROC-AUC={roc:.3f}  PR-AUC={pr:.3f}', fontsize=10)
    ax.set_xlabel('anomaly score'); ax.legend(fontsize=7)
    del det; gc.collect()

plt.suptitle('9.1: Score Distributions + Ranking Diagnostics', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'F9_score_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()

rank_df = pd.DataFrame(rank_rows).sort_values('pr_auc', ascending=False)
rank_df.to_csv(os.path.join(OUTPUT_DIR, 'topk_ranking_analysis.csv'), index=False)
print(rank_df.to_string(index=False))

In [ ]:
# 9.2 Subsampling ablation — does the 50K cap hurt SVM / KNN / LOF?
SUBSAMPLE_SIZES  = [10000, 20000, 50000, 80000, 120000]
SUBSAMPLE_MODELS = ['svm', 'knn', 'lof']
seed = 42; set_seed(seed)
X_tr, X_n, X_v, X_te, y_tr, y_v, y_te, _ = create_splits(df, FEATURE_COLS, CONFIG['TEST_SIZE'], CONFIG['VAL_SIZE'], seed)

sub_rows = []
for m in SUBSAMPLE_MODELS:
    print(f'\n  {m.upper()}')
    for n in SUBSAMPLE_SIZES:
        if n > len(X_n): continue
        rng = np.random.RandomState(seed)
        idx = rng.choice(len(X_n), n, replace=False)
        X_use = X_n[idx]
        t0 = time.time()
        det = create_model(m, len(FEATURE_COLS), seed=seed)
        det.train(X_use, None, X_v, y_v)
        yp = det.predict(X_te); ys = det.decision_scores(X_te)
        met = evaluate(y_te, yp, ys)
        met.update({'model': m, 'n': n, 'time': time.time()-t0})
        sub_rows.append(met)
        print(f'    n={n:<7} F1={met["f1"]:.4f}  recall={met["recall"]:.4f}  t={met["time"]:.1f}s')
        del det; gc.collect()

sub_df = pd.DataFrame(sub_rows)
sub_df.to_csv(os.path.join(OUTPUT_DIR, 'subsampling_ablation.csv'), index=False)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, m in zip(axes, SUBSAMPLE_MODELS):
    sm = sub_df[sub_df['model']==m]
    ax.plot(sm['n'], sm['f1'], '-o', label='F1'); ax.plot(sm['n'], sm['recall'], '-s', label='Recall')
    ax.axvline(50000, color='r', linestyle='--', alpha=0.5, label='50K cap')
    ax.set_title(m); ax.set_xlabel('n normal training samples'); ax.legend(); ax.grid(alpha=0.3)
plt.suptitle('F10: Subsampling Ablation', fontsize=14); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'F10_subsampling_ablation.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 9.3 Sensitivity under poisoning — tuned vs default at 10% targeted
# (We baked tuned defaults for PCA and SVM into the main grid. This reports the lift explicitly.)
print('='*70)
print('TUNED vs DEFAULT — 10% Targeted Poisoning (seed=42)')
print('='*70)

seed = 42; set_seed(seed)
X_tr, X_n, X_v, X_te, y_tr, y_v, y_te, _ = create_splits(df, FEATURE_COLS, CONFIG['TEST_SIZE'], CONFIG['VAL_SIZE'], seed)

def build_default(m, seed):
    c = CONFIG['CONTAMINATION']
    if m == 'pca': return PCA_AD(contamination=c, n_components=None, random_state=seed)
    if m == 'svm': return OCSVM(contamination=c, nu=0.05, gamma='scale')
    return None

rows = []
for m in ['pca', 'svm']:
    X_poisoned, _ = apply_poison(X_n, X_tr, y_tr, 'targeted_flip', 0.10, seed)
    X_use = subsample_if_heavy(X_poisoned, m, seed)

    # Default variant
    det_def = AnomalyDetector(m, pyod_model=build_default(m, seed), contamination=CONFIG['CONTAMINATION'], seed=seed)
    det_def.train(X_use, None, X_v, y_v)
    met_def = evaluate(y_te, det_def.predict(X_te), det_def.decision_scores(X_te))

    # Tuned variant (uses TUNED_PARAMS inside AnomalyDetector)
    det_tun = AnomalyDetector(m, contamination=CONFIG['CONTAMINATION'], seed=seed)
    det_tun.train(X_use, None, X_v, y_v)
    met_tun = evaluate(y_te, det_tun.predict(X_te), det_tun.decision_scores(X_te))

    rows.append({'model': m,
                 'default_f1': met_def['f1'], 'default_fnr': met_def['fnr'],
                 'tuned_f1':   met_tun['f1'], 'tuned_fnr':   met_tun['fnr'],
                 'delta_f1': met_tun['f1'] - met_def['f1']})
    print(f'{m:<6} default F1={met_def["f1"]:.4f}  tuned F1={met_tun["f1"]:.4f}  Δ={met_tun["f1"]-met_def["f1"]:+.4f}')
    del det_def, det_tun; gc.collect()

pd.DataFrame(rows).to_csv(os.path.join(OUTPUT_DIR, 'tuned_vs_default_poison.csv'), index=False)

---
## 10. Compute Cost & Multi-Criteria Ranking


In [ ]:
# 10.1 Compute-cost table
print('='*90)
print('COMPUTE COST TABLE')
print('='*90)
print(f'{"Model":<14} {"Clean F1":>10} {"Train(s)":>10} {"ParamCount":>12} {"Verdict":<20}')
print('─'*90)
cost_rows = []
for m in CONFIG['MODELS']:
    sub = clean_df[clean_df['model']==m]
    if len(sub) == 0: continue
    f1 = sub['f1'].mean(); t = sub['time'].mean()
    verdict = 'lightweight (<10s)' if t < 10 else ('moderate (<100s)' if t < 100 else 'heavy (>100s)')
    cost_rows.append({'model': m, 'clean_f1': f1, 'train_s': t, 'verdict': verdict})
    print(f'{m:<14} {f1:>10.4f} {t:>10.1f} {verdict:>20}')

pd.DataFrame(cost_rows).to_csv(os.path.join(OUTPUT_DIR, 'compute_cost.csv'), index=False)

In [ ]:
# 10.2 Multi-criteria robustness ranking
print('='*90)
print('MULTI-CRITERIA ROBUSTNESS RANKING')
print('='*90)

rank_rows = []
for m in CONFIG['MODELS']:
    clean_sub = clean_df[clean_df['model']==m]
    pois_sub  = poisoned_df[poisoned_df['model']==m]
    if len(clean_sub) == 0 or len(pois_sub) == 0: continue

    clean_f1 = clean_sub['f1'].mean()
    # Worst-case F1 drop across all attacks/rates
    worst_drop = clean_f1 - pois_sub.groupby(['attack','poison_rate'])['f1'].mean().min()
    worst_fnr  = pois_sub.groupby(['attack','poison_rate'])['fnr'].mean().max()
    seed_std   = pois_sub.groupby(['attack','poison_rate'])['f1'].std().mean()
    t = clean_sub['time'].mean()
    rank_rows.append({'model': m, 'clean_f1': clean_f1,
                      'worst_f1_drop': worst_drop, 'worst_fnr': worst_fnr,
                      'mean_seed_std': seed_std, 'train_s': t})

rank_df = pd.DataFrame(rank_rows)
# Composite score: reward clean F1, penalize worst drop, worst FNR, seed instability, runtime
# Normalize each criterion to [0,1] where 1=best
def norm(x, higher_better=True):
    lo, hi = x.min(), x.max()
    if hi == lo: return np.ones_like(x, dtype=float)
    y = (x - lo) / (hi - lo)
    return y if higher_better else (1 - y)

rank_df['score_clean']    = norm(rank_df['clean_f1'],      higher_better=True)
rank_df['score_robust']   = norm(rank_df['worst_f1_drop'], higher_better=False)
rank_df['score_safety']   = norm(rank_df['worst_fnr'],     higher_better=False)
rank_df['score_stability']= norm(rank_df['mean_seed_std'], higher_better=False)
rank_df['score_speed']    = norm(rank_df['train_s'],       higher_better=False)
weights = dict(clean=0.30, robust=0.25, safety=0.25, stability=0.10, speed=0.10)
rank_df['composite'] = (weights['clean']*rank_df['score_clean']
                       + weights['robust']*rank_df['score_robust']
                       + weights['safety']*rank_df['score_safety']
                       + weights['stability']*rank_df['score_stability']
                       + weights['speed']*rank_df['score_speed'])
rank_df = rank_df.sort_values('composite', ascending=False).reset_index(drop=True)
rank_df.to_csv(os.path.join(OUTPUT_DIR, 'multi_criteria_ranking.csv'), index=False)

print(f'\nWeights: {weights}\n')
print(rank_df[['model','clean_f1','worst_f1_drop','worst_fnr','mean_seed_std','train_s','composite']].to_string(index=False))

---
## 11. Save outputs


In [ ]:
import glob
print('All outputs saved to:', OUTPUT_DIR)
for f in sorted(glob.glob(os.path.join(OUTPUT_DIR, '**/*'), recursive=True)):
    if os.path.isfile(f):
        size = os.path.getsize(f)
        print(f'  {os.path.relpath(f, OUTPUT_DIR):<60} {size:>10,} bytes')

In [ ]:
# 11.2 Done — summary of deliverables for the paper
print('=' * 72)
print('PAPER RUN COMPLETE')
print('=' * 72)
print(f'Output folder: {OUTPUT_DIR}')
print()
print('Deliverables:')
print('  Tables:')
print(f'    - {OUTPUT_DIR}/table_T5_poisoning_impact.csv')
print(f'    - {OUTPUT_DIR}/compute_cost.csv')
print(f'    - {OUTPUT_DIR}/multi_criteria_ranking.csv')
print(f'    - {OUTPUT_DIR}/tuned_vs_default_poison.csv')
print('  Raw results:')
print(f'    - {OUTPUT_DIR}/all_results.csv')
print(f'    - {OUTPUT_DIR}/checkpoints/clean_baselines.csv')
print(f'    - {OUTPUT_DIR}/checkpoints/attack_checkpoint.csv')
print('  Figures:')
print(f'    - {OUTPUT_DIR}/figures/*.png')
print()
print('Sensitivity-analysis companion (Phase 1 + Phase 2) remains in')
print("  /content/drive/MyDrive/Adv-ML-final/v16_anomaly_diag/")
print('and should be cited in the paper appendix as hyperparameter justification.')
